# שיעור 12 - הפחתת היסטוריית הצ'אט עם לוח סיכות של סוכן

פנקס זה מציג כיצד לנהל הקשר בשיחות ארוכות באמצעות Microsoft Agent Framework. ככל שהשיחות מתארכות, מספר הטוקנים גדל — בסופו של דבר חורג מחלון ההקשר של המודל. אנו מתמודדים עם זה באמצעות **תבנית סיכום הקשר** ו**לוח סיכות של סוכן** לזיכרון ממושך.

## מה תלמדו:
1. **מדוע ניהול הקשר חשוב**: הבנת מגבלות טוקנים וחלונות הקשר
2. **סוכנים מודעי הקשר**: בניית סוכנים שמנהלים את הקשר של השיחה שלהם
3. **תבנית סיכום הקשר**: שימוש בכלים לדחיסת היסטוריית השיחה
4. **לוח הסיכות של הסוכן**: זיכרון מתמשך שחש בפני קיצוץ ההקשר

## דרישות מוקדמות:
- הגדרת Azure OpenAI עם משתני סביבה מוגדרים
- הבנה של מושגי סוכן בסיסיים משיעורים קודמים


## התקנה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## מדוע ניהול הקשר חשוב  

לכל LLM יש **חלון הקשר** סופי — מספר הטוקנים המקסימלי שהוא יכול לעבד בבקשה אחת. ככל שהשיחה הרב-סבבית מתקדמת:  

- **מספר הטוקנים גדל בקצב לינארי** עם כל הודעת משתמש ותשובת עוזר.  
- **טוקנים בפרומפט שולטים על העלות** כי כל ההיסטוריה נשלחת מחדש בכל סבב.  
- בסופו של דבר השיחה **חורגת מחלון ההקשר** והמודל או מקצר או נותן שגיאה.  

### אסטרטגיות לניהול הקשר  

| אסטרטגיה | איך היא פועלת | פשרה |  
|---|---|---|  
| **קיצור** | משליכים הודעות ישנות | מאבדים הקשר מוקדם |  
| **סיכום** | מתמצתים הודעות ישנות לסיכום | מפסידים חלק מהפרטים, אבל שומרים על הנקודות המרכזיות |  
| **מחברת / זכרון חיצוני** | שומרים עובדות מפתח מחוץ לשיחה | דורש קריאות כלים, אבל שורד כל קיצור |  

במחברת זו אנו משלבים **סיכום** עם **כלי מחברת** כך שהסוכן יוכל לשמור על רציפות גם כשהיסטוריית השיחה מתומצתת.  


## יצירת סוכן מודע להקשר


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## סימולציה של שיחה ארוכה

בואו נלך דרך שיחה עם מספר סבבים כדי לראות איך ההקשר מצטבר. הסוכן צריך לשמור על פרטים חשובים (העדפות, תקציב, תאריכי נסיעה) לאורך הסבבים ולהדגים רציפות.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

שימו לב כיצד הסוכן שומר על הקשר מהסבבים הקודמים — הוא יודע על יפן, סושי, מקדשים, צילום, תקציב של 3000$, נסיעה לבד, ומועד באפריל. בשיחה קצרה זה עובד היטב, אך ככל שהשיחה מתארכת היסטוריית השיחה המלאה נהיית יקרה לשליחה מחדש.

נמשיך את השיחה עם עוד סבבים כדי לראות את הצטברות ההקשר:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## תבנית סיכום הקשר

ככל שהשיחה מתארכת, ניתן להשתמש ב**כלי סיכום** לדחיסת ההקשר המצטבר לפורמט קומפקטי. הסוכן קורא לכלי זה כדי לתעד העדפות מפתח כך שגם אם הודעות ישנות יותר יורדו, המידע החיוני יישמר.

תבנית זו היא אבני הבניין לצמצום היסטוריה מורכב יותר:
1. הסוכן מזהה עובדות מפתח מהשיחה
2. הוא קורא לכלי הסיכום כדי לשמרן
3. ניתן להסיר בבטחה הודעות ישנות יותר כי הסיכום קולט את מה שחשוב

למטה אנו מגדירים כלי בשם `summarize_preferences` שהסוכן יכול לקרוא לו כדי לתעד סיכום קומפקטי של מה שלמד.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Summary

In this lesson you learned how to manage context in long-running agent conversations using Microsoft Agent Framework:

### Key Concepts
- **Context windows are finite** — every token in the conversation history costs money and counts toward the limit.
- **Summarization tools** let the agent condense accumulated context into compact summaries, reducing token usage while preserving essential information.
- **Agent scratchpads** provide persistent external memory that survives any conversation reduction.

### What You Built
- A **context-aware agent** that maintains continuity across multi-turn conversations
- A **summarization tool** (`summarize_preferences`) that records key user details in a compact format
- A **multi-turn conversation** demonstrating context retention and change handling

### Real-World Applications
- **Customer Service Bots**: Remember preferences across long support sessions
- **Personal Assistants**: Track ongoing projects without re-explaining context
- **Educational Tutors**: Maintain student progress across many interactions

### Next Steps
- Implement a full scratchpad tool with file-based persistence
- Add automatic history truncation after summarization
- Combine with vector databases for semantic memory search
- Build agents that can resume conversations days later with full context


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
